[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module0_TimeSeries/07_TimeAwareEvaluation.ipynb)

# Time-Aware Evaluation and Forecasting Baselines

**Module 0 · Lesson 7 of 13 · Student edition**  
**Estimated class time:** 75–90 minutes  
**Source sequence:** Original Day 3  

**Prerequisite:** Lesson 6  

## Learning objectives

By the end of this lesson, you should be able to:

- Create a chronological train/test split without leakage.
- Implement a naive last-value forecast.
- Fit an AR baseline using training data only.

---
## Part 0 — Imports & Setup

Run the cell below. You do **not** need to modify it.

In [ ]:
"""
Day 3 — Forecasting Models: From Baselines to Trees, MLPs, and Sequence Models
==============================================================================
Student Notebook

SLOs Covered:
  SLO 4 — Implement and evaluate multiple forecasting models (naive baseline, AR,
           tree-based model, MLP, RNN, and LSTM) using time-aware train/test splits.
  SLO 5 — Quantitatively compare forecasting performance using RMSE and MAE.
  SLO 6 — Explain hidden state, recurrence, and gating mechanisms in RNNs and LSTMs.

Dataset : Air Passengers (monthly, 1949–1960)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.ar_model import AutoReg
from statsmodels.graphics.tsaplots import plot_pacf

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

print('All imports successful!')

---
## Part 1 — Time-Aware Train/Test Splits

### Why can't we shuffle?

In standard supervised learning it is common to shuffle data before splitting into train and
test sets. **For time series, this is forbidden.** Shuffling breaks the temporal ordering
and causes **data leakage**: the model can be trained on observations from the *future*
relative to some test points, giving an artificially optimistic performance estimate.

The correct approach: split at a fixed point in time, keeping all training observations
*before* all test observations.

### 1.1 Load data (same pipeline as Days 1 & 2)

This cell is complete — run it.

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df  = pd.read_csv(url, parse_dates=['Month'])
df.columns = ['Date', 'Passengers']
df['Log_Passengers'] = np.log(df['Passengers'])
df['Log_Diff']       = df['Log_Passengers'].diff()

series = df['Log_Diff'].dropna().values
dates  = df['Date'].iloc[1:].values

print(f'Series length (after diff + dropna): {len(series)}')

In [ ]:
# Lag matrix helper — carried over from Day 2
def make_lag_matrix(series, n_lags):
    """
    Convert a 1-D time series into a lag-embedded feature matrix.

    Parameters
    ----------
    series : array-like, shape (T,)
    n_lags : int, number of lag features

    Returns
    -------
    X : np.ndarray, shape (T - n_lags, n_lags)
    y : np.ndarray, shape (T - n_lags,)
    """
    series = np.array(series)
    X, y = [], []
    for t in range(n_lags, len(series)):
        X.append(series[t - n_lags : t])
        y.append(series[t])
    return np.array(X), np.array(y)

### 1.2 Create the chronological split

**Your turn!** Build the lag matrix with `n_lags=12`, then split it into training (first 80%)
and test (last 20%) sets — **in order, without shuffling**.

> 💡 **Syntax reminder:** If `X` is a NumPy array, `X[:split]` and `X[split:]` give the
> first `split` rows and the remaining rows respectively.

In [ ]:
N_LAGS     = 12
TRAIN_FRAC = 0.80

X, y = make_lag_matrix(series, N_LAGS)

# FILL IN: compute the index on where to split (should be an integer, 80% of len(X))
split = ???

# FILL IN: slice X and y into training and test sets chronologically
X_train, X_test = X[???], X[???]
y_train, y_test = y[???], y[???]

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')

# Visualise the split
fig, ax = plt.subplots()
train_idx = np.arange(split)
test_idx  = np.arange(split, len(y))
ax.plot(train_idx, y_train, label='Train', color='steelblue')
ax.plot(test_idx,  y_test,  label='Test',  color='crimson')
ax.axvline(split, color='black', linestyle='--', linewidth=1.2, label='Split')
ax.set_title('Time-Aware Train / Test Split — Log-Differenced Passengers')
ax.legend()
plt.tight_layout()
plt.show()

### 💻 Coding Practice

1. Create a test/train function that takes in `TRAIN_FRAC`, `X`, and `y` as input and outputs the training data. Be sure to add appropriate docstring.

In [ ]:
# ADD Question for them to plot the models

---
## Part 2 — Naive Baseline & AR Model

### Why start with a baseline?

Every model evaluation begins with the **simplest possible forecast** — the baseline.
If a complex model can't beat it, the complexity is unjustified.

The **naive last-value baseline** (also called the *persistence forecast*) predicts that
the next value equals the last observed value:

$$\hat{x}_{t+1} = x_t$$

### 2.1 Implement the naive baseline

For each position in the test set, the naive prediction is the immediately preceding value.

In [ ]:
# FILL IN: naive baseline
# The first test prediction = last training value
# Each subsequent prediction = the previous actual test value
naive_preds = np.concatenate([[y_train[-1]], y_test[???]])

print('Naive baseline predictions (first 5):', naive_preds[:5])
print('Actual test values        (first 5):', y_test[:5])

### 2.2 Fit the AR model on training data

> 💡 **Reminder:** We must fit the AR model on **training data only** and use it to
> predict the test window. `AutoReg(series, lags=p).fit()` fits the model;
> `.predict(start, end)` generates forecasts at the original series indices.

In [ ]:
# Subset the series to training observations (include the burn-in lags)
train_series = series[:split + N_LAGS]

# FILL IN: 
ar_model  = ???
ar_result = ar_model.fit()

# Predict the test window (these indices refer to the original series)
ar_start = len(train_series)
ar_end   = ar_start + len(y_test) - 1
ar_preds = ar_result.predict(start=ar_start, end=ar_end, dynamic=False)

print('AR predictions (first 5):   ', ar_preds[:5])
print('Actual test values (first 5):', y_test[:5])